## Transformer  ->  GPT-1

## Transformer와 GPT-1 구조 변경 사항

1. Transformer의 Encoder-Decoder 구조를 Decoder-only 구조로 변경하였다.

2. Sin/Cos Positional Encoding 대신 학습 가능한 Position Embedding을 사용하였다.

3. Decoder의 Self-Attention만 사용하고 Encoder-Decoder Attention은 제거하였다.

4. 질문과 답변을 하나의 입력 시퀀스로 구성하여 Next Token Prediction 방식으로 학습하도록 Dataset을 수정하였다.

5. GPT의 Causal Mask를 적용하여 미래 토큰을 참조하지 못하도록 구현하였다.

In [1]:
!pip install gensim

In [2]:
!pip install nltk

In [3]:
!pip install sentencepiece

In [4]:
# ==========================================================
# 1. 라이브러리 import
# ==========================================================

# ----------------------------------------------------------
# 파일 및 폴더 경로 관리
# ----------------------------------------------------------
import os


# ----------------------------------------------------------
# 데이터 처리 라이브러리
# ----------------------------------------------------------
# DataFrame 형태로 CSV 데이터를 처리하기 위해 사용
import pandas as pd

# 행렬 연산 및 수치 계산을 위해 사용
import numpy as np


# ----------------------------------------------------------
# PyTorch 딥러닝 라이브러리
# ----------------------------------------------------------
# Tensor 연산 및 GPU 사용
import torch

# 신경망 Layer 구현
import torch.nn as nn

# Optimizer 구현
import torch.optim as optim


# ----------------------------------------------------------
# Dataset / DataLoader
# ----------------------------------------------------------
# 학습 데이터를 배치 단위로 전달하기 위해 사용
from torch.utils.data import Dataset, DataLoader


# ----------------------------------------------------------
# SentencePiece Tokenizer
# ----------------------------------------------------------
# 자연어 문장을 Token ID 형태로 변환하기 위해 사용
import sentencepiece as spm


# ----------------------------------------------------------
# Word2Vec
# ----------------------------------------------------------
# 데이터 증강(Lexical Substitution)을 위해 사용
from gensim.models import Word2Vec


# ----------------------------------------------------------
# 진행률 표시
# ----------------------------------------------------------
from tqdm import tqdm


# ----------------------------------------------------------
# 문자열 전처리
# ----------------------------------------------------------
import re


# ----------------------------------------------------------
# 랜덤 데이터 처리
# ----------------------------------------------------------
import random


# ----------------------------------------------------------
# BLEU Score 평가
# ----------------------------------------------------------
# 생성된 답변과 실제 답변의 유사도를 평가하기 위해 사용
from nltk.translate.bleu_score import sentence_bleu
from nltk.translate.bleu_score import SmoothingFunction

In [5]:
# ==========================================================
# 2. Random Seed 설정
# ==========================================================

# 동일한 결과 재현을 위해 난수 고정

SEED = 42

random.seed(SEED)

np.random.seed(SEED)

torch.manual_seed(SEED)


# GPU 사용 시 Seed 고정
if torch.cuda.is_available():

    torch.cuda.manual_seed_all(SEED)


print("Random Seed 설정 완료")

Random Seed 설정 완료


In [6]:
# ==========================================================
# 3. 학습 장치 설정
# ==========================================================

# GPU 사용 가능하면 CUDA 사용
# 불가능하면 CPU 사용

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)


print("현재 사용 장치 :", device)

현재 사용 장치 : cpu


In [7]:
# ==========================================================
# 4. 데이터 경로 설정
# ==========================================================


# Chatbot 데이터 CSV 경로

DATA_PATH = os.path.join(
    "data",
    "ChatbotData.csv"
)



# Word2Vec 사전학습 모델 경로

W2V_PATH = os.path.join(
    "data",
    "ko.bin"
)



print("데이터 경로 :", DATA_PATH)

print("Word2Vec 경로 :", W2V_PATH)

데이터 경로 : data/ChatbotData.csv
Word2Vec 경로 : data/ko.bin


In [8]:
# ==========================================================
# 5. 파일 존재 확인
# ==========================================================


if os.path.exists(DATA_PATH):

    print("ChatbotData.csv 파일 확인 완료")

else:

    print("ChatbotData.csv 파일을 찾을 수 없습니다.")



if os.path.exists(W2V_PATH):

    print("ko.bin 파일 확인 완료")

else:

    print("ko.bin 파일을 찾을 수 없습니다.")

ChatbotData.csv 파일 확인 완료
ko.bin 파일 확인 완료


In [9]:
# ==========================================================
# 6. 데이터 로드
# ==========================================================


# CSV 파일 읽기

data = pd.read_csv(DATA_PATH)



# 데이터 개수 확인

print("전체 데이터 개수 :", len(data))



# 데이터 상위 5개 확인

data.head()

전체 데이터 개수 : 11823


,Q,A,label
0,12시 땡!,하루가 또 가네요.,0
1,1지망 학교 떨어졌어,위로해 드립니다.,0
2,3박4일 놀러가고 싶다,여행은 언제나 좋죠.,0
3,3박4일 정도 놀러가고 싶다,여행은 언제나 좋죠.,0
4,PPL 심하네,눈살이 찌푸려지죠.,0


In [10]:
# ==========================================================
# 7. 데이터 정보 확인
# ==========================================================


# 컬럼 확인

print(data.columns)



# 결측치 확인

print("\n결측치 확인")

print(data.isnull().sum())



# 데이터 타입 확인

print("\n데이터 타입")

print(data.dtypes)

Index(['Q', 'A', 'label'], dtype='object')

결측치 확인
Q        0
A        0
label    0
dtype: int64

데이터 타입
Q        object
A        object
label     int64
dtype: object


In [11]:
# ==========================================================
# 8. 질문 / 답변 분리
# ==========================================================


questions = data["Q"].tolist()

answers = data["A"].tolist()



print("질문 개수 :", len(questions))

print("답변 개수 :", len(answers))



print("\n샘플 데이터 확인")

print("Q :", questions[0])

print("A :", answers[0])

질문 개수 : 11823
답변 개수 : 11823

샘플 데이터 확인
Q : 12시 땡!
A : 하루가 또 가네요.


In [12]:
# ==========================================================
# 9. 문장 전처리 함수
# ==========================================================


def preprocess_sentence(sentence):


    # ------------------------------------------------------
    # 1) 영어 대문자를 소문자로 변경
    # ------------------------------------------------------

    sentence = sentence.lower()



    # ------------------------------------------------------
    # 2) 앞뒤 공백 제거
    # ------------------------------------------------------

    sentence = sentence.strip()



    # ------------------------------------------------------
    # 3) 문장부호 앞뒤에 공백 추가
    # 예:
    # 안녕? → 안녕 ?
    # ------------------------------------------------------

    sentence = re.sub(
        r"([?.!,])",
        r" \1 ",
        sentence
    )



    # ------------------------------------------------------
    # 4) 여러 개의 공백을 하나로 변경
    # ------------------------------------------------------

    sentence = re.sub(
        r'[" "]+',
        " ",
        sentence
    )



    # ------------------------------------------------------
    # 5) 한글, 영어, 기본 문장부호만 유지
    # ------------------------------------------------------

    sentence = re.sub(
        r"[^ㄱ-ㅎㅏ-ㅣ가-힣a-zA-Z?.!,]+",
        " ",
        sentence
    )



    # 마지막 공백 제거

    sentence = sentence.strip()



    return sentence

In [13]:
# ==========================================================
# 10. 전처리 적용
# ==========================================================


data["Q"] = data["Q"].apply(
    preprocess_sentence
)


data["A"] = data["A"].apply(
    preprocess_sentence
)



print("전처리 완료!")



data.head()

전처리 완료!


,Q,A,label
0,시 땡 !,하루가 또 가네요 .,0
1,지망 학교 떨어졌어,위로해 드립니다 .,0
2,박 일 놀러가고 싶다,여행은 언제나 좋죠 .,0
3,박 일 정도 놀러가고 싶다,여행은 언제나 좋죠 .,0
4,ppl 심하네,눈살이 찌푸려지죠 .,0


In [14]:
# ==========================================================
# 11. 전처리 결과 확인
# ==========================================================


print("Q :", data["Q"][0])

print("A :", data["A"][0])

Q : 시 땡 !
A : 하루가 또 가네요 .


In [15]:
# ==========================================================
# 12. Random Data Augmentation 함수
# ==========================================================


# ----------------------------------------------------------
# Random Deletion
#
# 문장 속 일부 단어를 랜덤하게 삭제
#
# 예)
# 오늘 너무 피곤해요
#
# ↓
#
# 오늘 피곤해요
# ----------------------------------------------------------


def random_deletion(
    sentence,
    delete_prob=0.1
):

    words = sentence.split()


    # 너무 짧은 문장은 변경하지 않음

    if len(words) <= 2:

        return sentence


    result = []


    for word in words:


        # delete_prob 확률로 단어 삭제

        if random.random() > delete_prob:

            result.append(word)



    # 모든 단어가 삭제되는 상황 방지

    if len(result) == 0:

        return random.choice(words)



    return " ".join(result)

In [16]:
# ==========================================================
# 13. Random Swap 함수
# ==========================================================


def random_swap(sentence):

    words = sentence.split()


    if len(words) <= 2:

        return sentence


    idx1, idx2 = random.sample(
        range(len(words)),
        2
    )


    words[idx1], words[idx2] = (
        words[idx2],
        words[idx1]
    )


    return " ".join(words)

In [17]:
# ==========================================================
# 14. 데이터 증강 실행
# ==========================================================


print(
    "원본 데이터 개수 :",
    len(data)
)



# 첫 번째 증강 데이터

aug_data_1 = data.copy()



aug_data_1["Q"] = aug_data_1["Q"].apply(
    lambda x: random_deletion(x)
)



# 두 번째 증강 데이터

aug_data_2 = data.copy()



aug_data_2["Q"] = aug_data_2["Q"].apply(
    lambda x: random_swap(x)
)



print("데이터 증강 완료")

원본 데이터 개수 : 11823
데이터 증강 완료


In [18]:
# ==========================================================
# 15. 데이터 병합
# ==========================================================


data_augmented = pd.concat(
    [
        data,
        aug_data_1,
        aug_data_2
    ]
)



# 완전히 같은 질문과 답변 제거

data_augmented = (
    data_augmented
    .drop_duplicates(
        subset=[
            "Q",
            "A"
        ]
    )
    .reset_index(drop=True)
)



print(
    "증강 후 데이터 개수 :",
    len(data_augmented)
)

증강 후 데이터 개수 : 24389


In [19]:
# ==========================================================
# 16. 증강 데이터 확인
# ==========================================================


count = 0

for i in range(len(data)):
    original = data.iloc[i]["Q"]
    augmented = aug_data_1.iloc[i]["Q"]

    if original != augmented:
        print(f"[{i}]")
        print("원본 :", original)
        print("증강 :", augmented)
        print("-" * 60)

        count += 1
        if count == 10:
            break

[0]
원본 : 시 땡 !
증강 : 시 !
------------------------------------------------------------
[2]
원본 : 박 일 놀러가고 싶다
증강 : 박 놀러가고
------------------------------------------------------------
[3]
원본 : 박 일 정도 놀러가고 싶다
증강 : 박 일 놀러가고 싶다
------------------------------------------------------------
[8]
원본 : sns 시간낭비인 거 아는데 매일 하는 중
증강 : 시간낭비인 거 아는데 매일 하는 중
------------------------------------------------------------
[9]
원본 : sns 시간낭비인데 자꾸 보게됨
증강 : 자꾸 보게됨
------------------------------------------------------------
[14]
원본 : 가난한 자의 설움
증강 : 자의 설움
------------------------------------------------------------
[15]
원본 : 가만 있어도 땀난다
증강 : 있어도 땀난다
------------------------------------------------------------
[21]
원본 : 가스비 장난 아님
증강 : 가스비 아님
------------------------------------------------------------
[26]
원본 : 가족 있어 ?
증강 : 가족 있어
------------------------------------------------------------
[27]
원본 : 가족관계 알려 줘
증강 : 알려 줘
------------------------------------------------------------


In [20]:
same = (data["Q"] == aug_data_1["Q"]).sum()

print(f"같은 문장 수 : {same}")
print(f"전체 문장 수 : {len(data)}")
print(f"변경 비율 : {(1 - same/len(data))*100:.2f}%")

같은 문장 수 : 8417
전체 문장 수 : 11823
변경 비율 : 28.81%


In [21]:
# ==========================================================
# 17. SentencePiece 학습 데이터 생성
# ==========================================================


# SentencePiece 학습용 텍스트 파일 생성

SPM_INPUT_FILE = "chatbot_spm.txt"



with open(
    SPM_INPUT_FILE,
    "w",
    encoding="utf-8"
) as f:


    # ==========================================================
    # GPT-1 설명
    # ----------------------------------------------------------
    # GPT는 Decoder-only 모델이지만,
    # Tokenizer는 질문과 답변 전체 문장을 이용하여
    # Vocabulary를 학습한다.
    # 따라서 Q와 A를 모두 SentencePiece 학습 데이터에 포함한다.
    # ==========================================================

    # 질문 데이터 저장

    for sentence in data_augmented["Q"]:

        f.write(
            sentence + "\n"
        )



    # 답변 데이터 저장

    for sentence in data_augmented["A"]:

        f.write(
            sentence + "\n"
        )



print(
    "SentencePiece 학습 데이터 생성 완료"
)

SentencePiece 학습 데이터 생성 완료


In [22]:
# ==========================================================
# 18. SentencePiece Tokenizer 학습 (GPT-1)
# ==========================================================

VOCAB_SIZE = 8000

# ==========================================================
# GPT-1 변경
# ----------------------------------------------------------
# GPT는 Question과 Answer를 하나의 시퀀스로 학습한다.
# 이를 구분하기 위해 [SEP] 토큰을 SentencePiece의
# 사용자 정의 토큰(user_defined_symbols)으로 추가한다.
# ==========================================================

spm.SentencePieceTrainer.Train(

    f"--input={SPM_INPUT_FILE} "
    f"--model_prefix=chatbot_spm "
    f"--vocab_size={VOCAB_SIZE} "
    f"--pad_id=0 "
    f"--bos_id=1 "
    f"--eos_id=2 "
    f"--unk_id=3 "
    f"--user_defined_symbols=[SEP]"

)

print("GPT용 SentencePiece 학습 완료")

I0000 00:00:1784017368.747841    2415 sentencepiece_trainer.cc:227] Running command: --input=chatbot_spm.txt --model_prefix=chatbot_spm --vocab_size=8000 --pad_id=0 --bos_id=1 --eos_id=2 --unk_id=3 --user_defined_symbols=[SEP]
I0000 00:00:1784017368.750994    2415 sentencepiece_trainer.cc:105] Starts training with : 
trainer_spec {
  input: chatbot_spm.txt
  input_format: 
  model_prefix: chatbot_spm
  model_type: UNIGRAM
  vocab_size: 8000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  user_defined_symbols: [SEP]
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_sc

GPT용 SentencePiece 학습 완료


I0000 00:00:1784017374.854920    2415 unigram_model_trainer.cc:644] EM sub_iter=0 size=8800 obj=13.019 num_tokens=304001 num_tokens/piece=34.5456
I0000 00:00:1784017374.901749    2415 unigram_model_trainer.cc:644] EM sub_iter=1 size=8800 obj=12.973 num_tokens=305079 num_tokens/piece=34.6681
I0000 00:00:1784017374.909743    2415 trainer_interface.cc:702] Saving model: chatbot_spm.model
I0000 00:00:1784017374.914154    2415 trainer_interface.cc:714] Saving vocabs: chatbot_spm.vocab


Transformer에서는 Question -> Encoder  

GPT에서는 Question -> [SEP] -> Answer  

그래서 [SEP]를 Vocabulary에 등록

In [23]:
# ==========================================================
# 19. SentencePiece 모델 로드
# ==========================================================


sp = spm.SentencePieceProcessor()



sp.Load(
    "chatbot_spm.model"
)



print(
    "Tokenizer Load 완료"
)

Tokenizer Load 완료


In [24]:
# ==========================================================
# 20. Token 변환 테스트
# ==========================================================


test_sentence = "오늘 날씨 어때?"



# 문장 → Token

tokens = sp.encode_as_pieces(
    test_sentence
)



# 문장 → 숫자 ID

token_ids = sp.encode_as_ids(
    test_sentence
)



print(
    "원본 문장 :",
    test_sentence
)



print(
    "Token :",
    tokens
)



print(
    "Token ID :",
    token_ids
)


# ==========================================================
# GPT-1 변경
# ----------------------------------------------------------
# [SEP] 토큰이 정상적으로 등록되었는지 확인
# ==========================================================

print("[SEP] Token ID :", sp.piece_to_id("[SEP]"))

print("Token 확인 :", sp.id_to_piece(sp.piece_to_id("[SEP]")))

원본 문장 : 오늘 날씨 어때?
Token : ['▁오늘', '▁날씨', '▁어때', '?']
Token ID : [68, 604, 312, 7985]
[SEP] Token ID : 4
Token 확인 : [SEP]


In [25]:
# ==========================================================
# 21. Special Token 확인
# ==========================================================


print(
    "PAD ID :",
    sp.pad_id()
)


print(
    "BOS ID :",
    sp.bos_id()
)


print(
    "EOS ID :",
    sp.eos_id()
)


print(
    "UNK ID :",
    sp.unk_id()
)

PAD ID : 0
BOS ID : 1
EOS ID : 2
UNK ID : 3


In [26]:
# ==========================================================
# 22. ChatbotDataset 클래스 구현
# ==========================================================


class ChatbotDataset(Dataset):


    def __init__(
        self,
        dataframe,
        tokenizer,
        max_len=40
    ):


        # 데이터 저장

        self.data = dataframe.reset_index(
            drop=True
        )


        # SentencePiece tokenizer

        self.tokenizer = tokenizer


        # 최대 길이

        self.max_len = max_len



    def __len__(self):


        # 데이터 개수 반환

        return len(self.data)



    def __getitem__(
        self,
        idx
    ):


        # --------------------------------------
        # 질문(Q), 답변(A) 가져오기
        # --------------------------------------

        question = self.data.loc[
            idx,
            "Q"
        ]


        answer = self.data.loc[
            idx,
            "A"
        ]


        # ==========================================================
        # GPT-1 변경
        # ----------------------------------------------------------
        # GPT는 Encoder와 Decoder를 분리하지 않는다.
        # Question과 Answer를 하나의 시퀀스로 연결하여
        # 하나의 입력 시퀀스로 학습한다.
        #
        # SentencePiece에 [SEP] 토큰을 따로 학습시키지 않았으므로
        # Question과 Answer를 공백으로 연결한다.
        # ==========================================================

        text = question + " " + answer

        # --------------------------------------
        # GPT 입력 시퀀스 생성
        # --------------------------------------
        # <BOS>
        # Question
        # [SEP]
        # Answer
        # <EOS>
        #
        # GPT는 하나의 시퀀스로 학습한다.
        # --------------------------------------

        token_ids = (

            [self.tokenizer.bos_id()]

            +
    
            self.tokenizer.encode_as_ids(text)

            +

            [self.tokenizer.eos_id()]

        )



        # --------------------------------------
        # 길이 제한
        # --------------------------------------

        token_ids = token_ids[:self.max_len]

        # ==========================================================
        # GPT-1 변경
        # ----------------------------------------------------------
        # Next Token Prediction
        #
        # Input :
        # <BOS> 오늘 날씨 어때 [SEP] 맑아요
        #
        # Target :
        # 오늘 날씨 어때 [SEP] 맑아요 <EOS>
        #
        # 입력과 정답을 한 칸씩 Shift하여 생성한다.
        # ==========================================================

        input_ids = token_ids[:-1]

        target_ids = token_ids[1:]



        return (

            torch.tensor(
                input_ids,
                dtype=torch.long
            ),

            torch.tensor(
                target_ids,
                dtype=torch.long
            )

        )

GPT-1은 Encoder-Decoder 구조가 아닌 Decoder-only 구조를 사용하므로, 질문과 답변을 각각 입력하는 대신 하나의 시퀀스로 결합하였다. 또한 다음 토큰을 예측하는 자기회귀(Autoregressive) 학습을 위해 입력과 정답 시퀀스를 한 토큰씩 이동(Shift)하여 구성하였다.

In [27]:
# ==========================================================
# 23. Dataset 생성 테스트
# ==========================================================


MAX_LEN = 40



dataset = ChatbotDataset(
    data_augmented,
    sp,
    MAX_LEN
)



print(
    "Dataset 크기 :",
    len(dataset)
)



src, trg = dataset[0]



print(
    "Input IDs :",
    src
)


print(
    "Target IDs :",
    trg
)

Dataset 크기 : 24389
Input IDs : tensor([   1,  586,    5, 7404,    5, 7931, 2654,    5, 7898, 3434,    6])
Target IDs : tensor([ 586,    5, 7404,    5, 7931, 2654,    5, 7898, 3434,    6,    2])


In [28]:
# ==========================================================
# 24. Padding 처리 함수
# ==========================================================


def collate_fn(batch):


    # ==========================================================
    # GPT-1 변경
    # ----------------------------------------------------------
    # batch 안에는
    #
    # (
    #   input_ids,
    #   target_ids
    # )
    #
    # 형태의 데이터가 들어있다.
    # ==========================================================

    # ==========================================================
    # GPT-1 변경
    # ----------------------------------------------------------
    # GPT는 Encoder/Decoder 입력을 사용하지 않는다.
    # 하나의 입력 시퀀스(Input)와 정답(Target)을 Padding한다.
    # ==========================================================

    input_batch = []

    target_batch = []



    for input_ids, target_ids in batch:

        input_batch.append(input_ids)

        target_batch.append(target_ids)



    # --------------------------------------
    # Padding 적용
    #
    # batch_first=True
    #
    # 결과 형태:
    #
    # (batch_size, sequence_length)
    # --------------------------------------


    input_batch = torch.nn.utils.rnn.pad_sequence(

        input_batch,

        batch_first=True,

        padding_value=sp.pad_id()

    )



    target_batch = torch.nn.utils.rnn.pad_sequence(

        target_batch,

        batch_first=True,

        padding_value=sp.pad_id()

    )



    return (

        input_batch,

        target_batch

    )

In [29]:
# ==========================================================
# 25. DataLoader 생성
# ==========================================================


BATCH_SIZE = 64



dataloader = DataLoader(

    dataset,

    batch_size=BATCH_SIZE,

    shuffle=True,

    collate_fn=collate_fn

)



print(
    "DataLoader 생성 완료"
)

DataLoader 생성 완료


In [30]:
# ==========================================================
# 26. Batch 데이터 확인
# ==========================================================


input_batch, target_batch = next(
    iter(dataloader)
)



print(
    "Input IDs Shape:"
)

print(
    input_batch.shape
)



print(
    "\nTarget IDs Shape:"
)

print(
    target_batch.shape
)

Input IDs Shape:
torch.Size([64, 36])

Target IDs Shape:
torch.Size([64, 36])


In [31]:
# ==========================================================
# 27. Padding Token 확인
# ==========================================================


print(
    input_batch[0]
)


print(
    "PAD ID :",
    sp.pad_id()
)

tensor([   1, 2828,    5, 7917,  157,    5, 7974,  189,    9,  541, 2113,    5,
        7923, 7929,  207,    6,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0])
PAD ID : 0


In [32]:
# ==========================================================
# 28. Transformer Hyperparameter 설정
# ==========================================================


# SentencePiece Vocabulary 크기

VOCAB_SIZE = sp.get_piece_size()



# 문장 최대 길이

MAX_LEN = 40



# Batch 크기

BATCH_SIZE = 64



# Embedding 차원

D_MODEL = 256



# Encoder / Decoder Layer 개수

N_LAYERS = 2



# Multi Head 개수

N_HEADS = 8



# Feed Forward 차원

D_FF = 512



# Dropout 비율

DROPOUT = 0.1



print(
    "Vocabulary Size :",
    VOCAB_SIZE
)

Vocabulary Size : 8000


In [33]:
# ==========================================================
# 29. Position Embedding (GPT-1)
# ==========================================================

class PositionEmbedding(nn.Module):

    def __init__(
        self,
        max_len,
        d_model
    ):

        super().__init__()

        # ==========================================================
        # GPT-1 변경
        # ----------------------------------------------------------
        # Transformer는 Sin/Cos Positional Encoding을 사용하지만,
        # GPT-1은 학습 가능한 Position Embedding을 사용한다.
        # ==========================================================

        self.position_embedding = nn.Embedding(
            max_len,
            d_model
        )


    def forward(
        self,
        x
    ):

        batch_size = x.size(0)

        seq_len = x.size(1)

        positions = torch.arange(
            seq_len,
            device=x.device
        ).unsqueeze(0).expand(
            batch_size,
            seq_len
        )

        return self.position_embedding(
            positions
        )

In [34]:
# ==========================================================
# 30. Multi-Head Attention
# ==========================================================


class MultiHeadAttention(nn.Module):


    def __init__(
        self,
        d_model,
        n_heads
    ):


        super().__init__()



        assert d_model % n_heads == 0



        self.d_model = d_model

        self.n_heads = n_heads


        self.head_dim = (
            d_model //
            n_heads
        )



        # Query, Key, Value Linear

        self.q_linear = nn.Linear(
            d_model,
            d_model
        )


        self.k_linear = nn.Linear(
            d_model,
            d_model
        )


        self.v_linear = nn.Linear(
            d_model,
            d_model
        )



        # 최종 출력 Linear

        self.fc_out = nn.Linear(
            d_model,
            d_model
        )



    def forward(
        self,
        query,
        key,
        value,
        mask=None
    ):


        batch_size = query.shape[0]



        Q = self.q_linear(query)

        K = self.k_linear(key)

        V = self.v_linear(value)



        # Head 분리

        Q = Q.view(
            batch_size,
            -1,
            self.n_heads,
            self.head_dim
        ).permute(
            0,2,1,3
        )


        K = K.view(
            batch_size,
            -1,
            self.n_heads,
            self.head_dim
        ).permute(
            0,2,1,3
        )


        V = V.view(
            batch_size,
            -1,
            self.n_heads,
            self.head_dim
        ).permute(
            0,2,1,3
        )



        # Attention Score 계산

        energy = torch.matmul(

            Q,

            K.permute(
                0,
                1,
                3,
                2
            )

        ) / np.sqrt(
            self.head_dim
        )



        # Mask 적용

        if mask is not None:

            energy = energy.masked_fill(
                mask == 0,
                -1e10
            )



        attention = torch.softmax(
            energy,
            dim=-1
        )



        out = torch.matmul(
            attention,
            V
        )



        # Head 합치기

        out = out.permute(
            0,
            2,
            1,
            3
        ).contiguous()



        out = out.view(
            batch_size,
            -1,
            self.d_model
        )



        return self.fc_out(out)

In [35]:
# ==========================================================
# 31-1. Feed Forward Network
# ==========================================================


class FeedForward(nn.Module):


    def __init__(
        self,
        d_model,
        d_ff,
        dropout=0.1
    ):


        super().__init__()



        self.linear1 = nn.Linear(
            d_model,
            d_ff
        )


        self.linear2 = nn.Linear(
            d_ff,
            d_model
        )


        self.dropout = nn.Dropout(
            dropout
        )


        # ==========================================================
        # GPT-1 변경
        # ----------------------------------------------------------
        # GPT-1에서는 Feed Forward Network의 활성화 함수로
        # ReLU 대신 GELU를 사용한다.
        # ==========================================================

        self.gelu = nn.GELU()



    def forward(
        self,
        x
    ):


        return self.linear2(

            self.dropout(

                self.gelu(

                    self.linear1(x)

                )

            )

        )

In [36]:
# ==========================================================
# 31-2. Encoder Layer
# ==========================================================

# ==========================================================
# GPT-1 변경
# ----------------------------------------------------------
# GPT는 Encoder를 사용하지 않는 Decoder-only 구조이다.
# 아래 EncoderLayer는 기존 Transformer 구현이며,
# GPT 모델에서는 사용하지 않는다.
# ==========================================================

class EncoderLayer(nn.Module):


    def __init__(
        self,
        d_model,
        n_heads,
        d_ff,
        dropout
    ):


        super().__init__()



        self.self_attn = MultiHeadAttention(
            d_model,
            n_heads
        )



        self.ffn = FeedForward(
            d_model,
            d_ff,
            dropout
        )



        self.norm1 = nn.LayerNorm(
            d_model
        )


        self.norm2 = nn.LayerNorm(
            d_model
        )



        self.dropout = nn.Dropout(
            dropout
        )



    def forward(
        self,
        src,
        src_mask
    ):


        # Self Attention

        attn_out = self.self_attn(

            src,
            src,
            src,
            src_mask

        )



        # Residual + Normalization

        src = self.norm1(

            src

            +

            self.dropout(attn_out)

        )



        # Feed Forward

        ffn_out = self.ffn(
            src
        )



        src = self.norm2(

            src

            +

            self.dropout(ffn_out)

        )



        return src

In [37]:
# ==========================================================
# 32. GPT Block (GPT-1)
# ==========================================================

class GPTBlock(nn.Module):

    def __init__(
        self,
        d_model,
        n_heads,
        d_ff,
        dropout
    ):

        super().__init__()

        # ==========================================================
        # GPT-1 변경
        # ----------------------------------------------------------
        # GPT는 Decoder-only 구조이므로
        # Masked Self Attention만 사용한다.
        # Encoder-Decoder Attention은 제거한다.
        # ==========================================================

        self.self_attn = MultiHeadAttention(
            d_model,
            n_heads
        )

        self.ffn = FeedForward(
            d_model,
            d_ff,
            dropout
        )

        self.norm1 = nn.LayerNorm(
            d_model
        )

        self.norm2 = nn.LayerNorm(
            d_model
        )

        self.dropout = nn.Dropout(
            dropout
        )

    def forward(
        self,
        x,
        mask
    ):

        # ==========================================================
        # GPT-1 변경
        # ----------------------------------------------------------
        # 1. Masked Self Attention
        # ==========================================================

        attn = self.self_attn(
            x,
            x,
            x,
            mask
        )

        x = self.norm1(
            x +
            self.dropout(attn)
        )

        # ==========================================================
        # GPT-1 변경
        # ----------------------------------------------------------
        # 2. Feed Forward Network
        # ==========================================================

        ffn = self.ffn(x)

        x = self.norm2(
            x +
            self.dropout(ffn)
        )

        return x

In [38]:
# ==========================================================
# 33. GPT Model
# ==========================================================

class GPTModel(nn.Module):


    def __init__(self):


        super().__init__()


        # ==========================================================
        # Token Embedding
        # ==========================================================

        self.embedding = nn.Embedding(

            VOCAB_SIZE,

            D_MODEL

        )


        # ==========================================================
        # GPT-1 변경
        # ----------------------------------------------------------
        # Transformer의 Sin/Cos Position Encoding 대신
        # 학습 가능한 Position Embedding 사용
        # ==========================================================

        self.position_embedding = PositionEmbedding(

            MAX_LEN,

            D_MODEL

        )


        self.dropout = nn.Dropout(

            DROPOUT

        )


        # ==========================================================
        # GPT-1 변경
        # ----------------------------------------------------------
        # Transformer의 Encoder와 Decoder를 제거하고
        # Decoder-only GPT Block만 사용
        # ==========================================================

        self.gpt_blocks = nn.ModuleList(

            [

                GPTBlock(

                    D_MODEL,

                    N_HEADS,

                    D_FF,

                    DROPOUT

                )

                for _ in range(N_LAYERS)

            ]

        )


        # ==========================================================
        # 출력층
        # ==========================================================

        self.fc_out = nn.Linear(

            D_MODEL,

            VOCAB_SIZE

        )

            # ==========================================================
    # GPT-1 변경
    # ----------------------------------------------------------
    # GPT는 미래 단어를 볼 수 없도록
    # Causal Mask(Upper Triangle Mask)를 사용한다.
    # ==========================================================
    def make_trg_mask(
        self,
        input_ids
    ):

        seq_len = input_ids.size(1)

        mask = torch.tril(

            torch.ones(

                seq_len,

                seq_len,

                device=input_ids.device

            )

        ).bool()

        return mask.unsqueeze(0).unsqueeze(0)



    # ==========================================================
    # GPT-1 Forward
    # ----------------------------------------------------------
    # 입력 :
    # input_ids
    #
    # 출력 :
    # 다음 Token의 Vocabulary Score(Logits)
    # ==========================================================
    def forward(
        self,
        input_ids
    ):


        # ------------------------------------------------------
        # Causal Mask 생성
        # ------------------------------------------------------

        trg_mask = self.make_trg_mask(
            input_ids
        )


        # ------------------------------------------------------
        # Token Embedding
        # ------------------------------------------------------

        token_embedding = self.embedding(
            input_ids
        )


        # ------------------------------------------------------
        # Position Embedding
        # ------------------------------------------------------

        position_embedding = self.position_embedding(
            input_ids
        )


        # ==========================================================
        # GPT-1 변경
        # ----------------------------------------------------------
        # Token Embedding + Position Embedding
        # ==========================================================

        x = token_embedding + position_embedding

        x = self.dropout(x)


        # ------------------------------------------------------
        # GPT Block 반복
        # ------------------------------------------------------

        for block in self.gpt_blocks:

            x = block(

                x,

                trg_mask

            )


        # ------------------------------------------------------
        # Vocabulary Projection
        # ------------------------------------------------------

        output = self.fc_out(

            x

        )


        return output

In [39]:
# ==========================================================
# 33. 모델 생성 테스트
# ==========================================================


device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)



# ==========================================================
# GPT-1 변경
# ----------------------------------------------------------
# TransformerChatbot 대신 GPTModel 생성
# ==========================================================

model = GPTModel().to(device)

print(model)

GPTModel(
  (embedding): Embedding(8000, 256)
  (position_embedding): PositionEmbedding(
    (position_embedding): Embedding(40, 256)
  )
  (dropout): Dropout(p=0.1, inplace=False)
  (gpt_blocks): ModuleList(
    (0-1): 2 x GPTBlock(
      (self_attn): MultiHeadAttention(
        (q_linear): Linear(in_features=256, out_features=256, bias=True)
        (k_linear): Linear(in_features=256, out_features=256, bias=True)
        (v_linear): Linear(in_features=256, out_features=256, bias=True)
        (fc_out): Linear(in_features=256, out_features=256, bias=True)
      )
      (ffn): FeedForward(
        (linear1): Linear(in_features=256, out_features=512, bias=True)
        (linear2): Linear(in_features=512, out_features=256, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (gelu): GELU(approximate='none')
      )
      (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropou

In [40]:
# ==========================================================
# 34. Loss 함수 / Optimizer 설정
# ==========================================================


# Padding Token

PAD_IDX = sp.pad_id()



# Cross Entropy Loss

criterion = nn.CrossEntropyLoss(

    ignore_index=PAD_IDX

)



# Optimizer

optimizer = optim.Adam(

    model.parameters(),

    lr=0.0001,

    betas=(0.9,0.98),

    eps=1e-9

)



print(
    "Loss / Optimizer 설정 완료"
)

Loss / Optimizer 설정 완료


In [41]:
# ==========================================================
# 35. GPT Training Loop
# ==========================================================

def train_epoch(
    model,
    dataloader,
    optimizer,
    criterion,
    device
):

    model.train()

    total_loss = 0

    for input_ids, target_ids in tqdm(
        dataloader
    ):

        # --------------------------------------------------
        # GPU 이동
        # --------------------------------------------------

        input_ids = input_ids.to(device)

        target_ids = target_ids.to(device)

        optimizer.zero_grad()

        # --------------------------------------------------
        # GPT-1 변경
        # --------------------------------------------------
        # Decoder-only 입력
        #
        # Input :
        # <BOS> 오늘 날씨 어때 맑아요
        #
        # Output :
        # 오늘 날씨 어때 맑아요 <EOS>
        # --------------------------------------------------

        output = model(
            input_ids
        )

        # --------------------------------------------------
        # [batch, seq_len, vocab]
        # ->
        # [batch * seq_len, vocab]
        # --------------------------------------------------

        output_dim = output.shape[-1]

        output = output.contiguous().view(
            -1,
            output_dim
        )

        # --------------------------------------------------
        # 정답도 동일하게 펼치기
        # --------------------------------------------------

        target_ids = target_ids.contiguous().view(
            -1
        )

        # --------------------------------------------------
        # Loss 계산
        # --------------------------------------------------

        loss = criterion(
            output,
            target_ids
        )

        # --------------------------------------------------
        # Backpropagation
        # --------------------------------------------------

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1
        )

        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

In [42]:
# ==========================================================
# 36. Model Training
# ==========================================================


EPOCHS = 5



for epoch in range(EPOCHS):


    loss = train_epoch(

        model,

        dataloader,

        optimizer,

        criterion,

        device

    )



    print(

        f"Epoch {epoch+1}/{EPOCHS}"

        f"  Loss : {loss:.4f}"

    )

100%|██████████| 382/382 [04:31<00:00,  1.41it/s]


Epoch 1/5  Loss : 6.4733


100%|██████████| 382/382 [04:29<00:00,  1.42it/s]


Epoch 2/5  Loss : 5.8123


100%|██████████| 382/382 [04:31<00:00,  1.41it/s]


Epoch 3/5  Loss : 5.5889


100%|██████████| 382/382 [04:29<00:00,  1.42it/s]


Epoch 4/5  Loss : 5.4117


100%|██████████| 382/382 [04:32<00:00,  1.40it/s]

Epoch 5/5  Loss : 5.2498


In [43]:
# ==========================================================
# GPT-1 변경
# ----------------------------------------------------------
# 37. GPT 모델 저장
# ==========================================================

MODEL_PATH = "chatbot_gpt.pt"

torch.save(
    model.state_dict(),
    MODEL_PATH
)

print(
    "모델 저장 완료:",
    MODEL_PATH
)

모델 저장 완료: chatbot_gpt.pt


In [44]:
# ==========================================================
# GPT-1 변경
# ----------------------------------------------------------
# 38. GPT 모델 불러오기
# ==========================================================

MODEL_PATH = "chatbot_gpt.pt"

model.load_state_dict(
    torch.load(
        MODEL_PATH,
        map_location=device
    )
)

model.to(device)

model.eval()

print(
    "모델 로드 완료"
)

모델 로드 완료


In [45]:
# ==========================================================
# 39. GPT 응답 생성 함수
# ==========================================================

def chatbot_response(sentence):

    model.eval()

    # ----------------------------------
    # 입력 문장 전처리
    # ----------------------------------
    sentence = preprocess_sentence(sentence)

    # ==========================================================
    # GPT-1 변경
    # ----------------------------------------------------------
    # GPT는 하나의 입력 시퀀스를 사용한다.
    # ==========================================================

    input_ids = (
        [sp.bos_id()]
        +
        sp.encode_as_ids(sentence)
    )

    input_ids = torch.tensor(
        input_ids,
        dtype=torch.long
    ).unsqueeze(0).to(device)

    # 최대 길이 제한
    input_ids = input_ids[:, :MAX_LEN]

    with torch.no_grad():

        # GPT-1 변경
        # 현재 입력 길이를 제외한 만큼만 생성
        for _ in range(MAX_LEN - input_ids.size(1)):

            output = model(input_ids)

            next_token = output[:, -1, :].argmax(
                dim=-1
            ).item()

            # EOS면 종료
            if next_token == sp.eos_id():
                break

            # 생성된 토큰 이어 붙이기
            next_token_tensor = torch.tensor(
                [[next_token]],
                device=device
            )

            input_ids = torch.cat(
                [
                    input_ids,
                    next_token_tensor
                ],
                dim=1
            )

    # ==========================================================
    # GPT-1 변경
    # ----------------------------------------------------------
    # 생성된 Token을 문장으로 복원한다.
    # BOS Token은 제거하고 문자열로 변환한다.
    # ==========================================================

    generated_ids = input_ids.squeeze().tolist()

    # BOS 제거
    generated_ids = generated_ids[1:]

    result = sp.decode_ids(generated_ids)

    return result.strip()

In [46]:
# ==========================================================
# 40. Chatbot 테스트
# ==========================================================


test_questions = [

    "오늘 날씨 어때?",

    "너무 피곤해",

    "영화 보고 싶어",

    "기분이 안 좋아"

]



for q in test_questions:


    answer = chatbot_response(q)


    print(
        "Q:",
        q
    )


    print(
        "A:",
        answer
    )


    print("-"*40)

Q: 오늘 날씨 어때?
A: 오늘 날씨 어때 ? ? ? ? ? 잘 ?
----------------------------------------
Q: 너무 피곤해
A: 너무 피곤해 ? ? ? ? ? ? 잘 ?
----------------------------------------
Q: 영화 보고 싶어
A: 영화 보고 싶어 ? ? ? ? ? 잘 ?
----------------------------------------
Q: 기분이 안 좋아
A: 기분이 안 좋아 ? ? ? ? ? ?
----------------------------------------


In [47]:
# ==========================================================
# 41. BLEU Score 평가
# ==========================================================


def calculate_bleu(
    dataframe,
    sample_num=100
):


    smoothie = SmoothingFunction().method4



    scores = []



    sample_data = dataframe.sample(

        n=min(

            sample_num,

            len(dataframe)

        ),

        random_state=42

    )



    for _, row in tqdm(

        sample_data.iterrows(),

        total=len(sample_data)

    ):



        question = row["Q"]

        reference = row["A"]



        prediction = chatbot_response(

            question

        )



        reference_tokens = reference.split()

        prediction_tokens = prediction.split()



        score = sentence_bleu(

            [reference_tokens],

            prediction_tokens,

            smoothing_function=smoothie

        )



        scores.append(score)



    return np.mean(scores)

In [ ]:
# ==========================================================
# BLEU Score 출력
# ==========================================================


bleu = calculate_bleu(

    data_augmented,

    sample_num=50

)



print(

    f"평균 BLEU Score : {bleu:.4f}"

)

 66%|██████▌   | 33/50 [00:02<00:01, 13.94it/s]

In [ ]:
# ==========================================================
# GPT 챗봇 테스트
# ==========================================================

questions = [
    "안녕",
    "오늘 기분이 어때?",
    "배고파",
    "영화 추천해줘",
    "잘자"
]

for q in questions:

    print(f"질문 : {q}")

    answer = chatbot_response(q)

    print(f"답변 : {answer}")

    print("-" * 50)


while True:

    question = input("질문 : ")

    if question.lower() == "exit":
        break

    answer = chatbot_response(question)

    print("답변 :", answer)

### 회고

이번 프로젝트에서는 기존 Transformer 기반 챗봇을 GPT-1 구조로 변경하는 과정을 수행하였다. 가장 큰 변화는 Encoder-Decoder 구조를 Decoder-only 구조로 변경하고, 입력 데이터를 Next Token Prediction 방식에 맞게 전처리한 점이었다. 또한 Sin/Cos Positional Encoding 대신 학습 가능한 Position Embedding을 적용하고, GPT의 Causal Mask를 구현하면서 Transformer와 GPT의 구조적 차이를 직접 확인할 수 있었다.

구현 과정에서는 Dataset과 Training Loop, 모델 구조가 서로 연결되어 있어 하나의 부분만 수정해서는 정상적으로 동작하지 않는다는 점을 경험하였다. 특히 입력 형식과 모델의 forward 함수, 학습 과정이 모두 일관된 구조를 가져야 한다는 점을 배울 수 있었다.

과제 제출 시간이 타이트해서 학습을 많이 시키지 못하고 5번으로 제한해서 진행했지만,  
학습을 진행하면서 Loss가 감소하는 것을 확인했고, 학습이 완료된 후에는 입력 문장에 대한 응답이 생성되는 것을 확인하였다. 비록 생성된 문장의 품질은 충분하지 않았지만, GPT 모델이 다음 토큰을 예측하는 방식으로 동작하는 과정을 직접 구현해 볼 수 있었던 점이 의미 있는 경험이었다.

이번 과제를 통해 Transformer와 GPT의 구조적 차이뿐만 아니라, 생성형 언어모델의 입력 구성과 학습 방식에 대해 이해를 높일 수 있었다. 앞으로는 더 큰 데이터셋과 충분한 학습을 통해 자연스러운 문장을 생성하는 모델을 구현해 보고 싶다.
